In [8]:
import pandas as pd
import numpy as np

In [1]:
from models.chemprop_3Dligand_affinity.config import load_experiment_config
from models.chemprop_3Dligand_affinity.data import load_data_splits
from chemprop import data, featurizers

Figuring out lmdb stuff

In [ ]:
fp = "processed/pose_manifest.csv"
df = pd.read_csv(fp)

In [4]:
print(df.columns)
df['pose_id'][0]

Index(['pose_id', 'pose_hash', 'pdb_key', 'ligand', 'grid', 'uniprot_id',
       'pdb_id', 'glide_score', 'pose_rank', 'is_top_rank', 'source_sdf'],
      dtype='str')


'2YDO_P29274_Brc1cccc(Nc2nc3c(N4CCCC4)ncnc3s2)c1_16e6e598bd01add0'

In [ ]:
# obj ends up being a pyg graph object. So we just need the pose_id to 
# get the pyg objects.

import lmdb
import pickle

path = "thndr_pose_processed/pose_lmdb"

env = lmdb.open(
    path,
    readonly=True,
    lock=False,
    readahead=False,
)
with env.begin() as txn:
    value = txn.get(b"2YDO_P29274_Brc1cccc(Nc2nc3c(N4CCCC4)ncnc3s2)c1_16e6e598bd01add0")

obj = pickle.loads(value)
print(obj)

Figuring out chemprop adding atom level features

In [ ]:
config = load_experiment_config()
df_filtered, train_indices, val_indices, test_indices = load_data_splits(config) 
smis = df_filtered.loc[:, 'ligand'].values
ys = df_filtered.loc[:, 'affinity'].values
all_data = [data.MoleculeDatapoint.from_smi(smi, np.array([y], dtype=np.float32)) 
    for smi, y in zip(smis, ys)]

train_data, val_data, test_data = data.split_data_by_indices(
all_data, [train_indices], [val_indices], [test_indices]
)

featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer() # TODO add atom coordinates

train_dset = data.MoleculeDataset(train_data[0], featurizer)
scaler = train_dset.normalize_targets()
val_dset = data.MoleculeDataset(val_data[0], featurizer)
val_dset.normalize_targets(scaler)
test_dset = data.MoleculeDataset(test_data[0], featurizer)

train_loader = data.build_dataloader(train_dset)
val_loader = data.build_dataloader(val_dset, shuffle=False)
test_loader = data.build_dataloader(test_dset, shuffle=False)

In [18]:
smi = 'CC(N)c1ccc(Cl)cc1'
dp = data.MoleculeDatapoint.from_smi(smi, 1)
n_atoms = dp.mol.GetNumAtoms()
V_f = np.random.randn(n_atoms, 3) # coords
dp.V_f = V_f

In [20]:
dp

MoleculeDatapoint(mol=<rdkit.Chem.rdchem.Mol object at 0x2a6fe6420>, y=1, weight=1.0, gt_mask=None, lt_mask=None, x_d=None, x_phase=None, name='CC(N)c1ccc(Cl)cc1', V_f=array([[ 1.42030828, -1.68378859,  0.88088784],
       [-1.10586311, -0.01943637, -0.7967382 ],
       [ 0.90498108, -1.32700513, -1.49144836],
       [ 0.38058807,  0.54278174,  0.54932671],
       [-0.51524109, -0.81291485, -0.8518125 ],
       [-0.39763536,  0.47541805, -0.4338421 ],
       [-0.6385288 ,  0.5107417 , -0.50211482],
       [ 0.15099892, -0.30325331,  0.2041801 ],
       [-0.3219202 ,  0.16631558, -0.39374999],
       [ 1.57321914,  0.8421604 ,  1.98167398]]), E_f=None, V_d=None)